# Studio · Compute로 경험하는 MLOps
데이터 버전 → 파이프라인 → MLflow → 품질 게이트 → 모델 등록 → 배포 → 재학습 → 롤백.

**실행 위치:** private Storage에 접근할 수 있는 `ci-mlops-private`. **Kernel:** AML MLOps Lab (Python 3.12).
처음 실행한다면 [Compute Instance 실행 준비](../docs/setup.md#compute-instance에서-실행-준비)에서 kernel·`config.json`·CLI 로그인을 준비하세요. 강사용 Azure 리소스 생성 단계를 다시 수행할 필요는 없습니다.

**Run all 대신 셀을 순서대로 실행합니다.** 의도적으로 실패하는 품질 게이트 셀이 있습니다. 배포 셀은 실제 Azure 비용을 발생시킵니다.

In [ ]:
import os
import sys
from pathlib import Path
from datetime import datetime, timezone

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file())
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from mlops_lab.config import Settings
from mlops_lab.data import make_dataset
from mlops_lab.pipeline import register_assets
from mlops_lab.operations import (
    submit, wait_for_run, snapshot_run, download_report, register_model,
    deploy, set_traffic, invoke, cleanup_runtime,
)

settings = Settings.load()
client = settings.client()
lab_id = datetime.now(timezone.utc).strftime('%Y%m%d%H%M%S')
baseline_run, bad_run, retrain_run = [f'{name}-{lab_id}' for name in ('baseline', 'bad', 'retrain')]
version1, version2, rejected_version = [lab_id + suffix for suffix in ('1', '2', '9')]
print(settings.workspace, settings.compute_instance, settings.compute_cluster)
print('Run suffix:', lab_id)

## 1. 데이터·환경·컴포넌트
v1은 학습 800행 + 검증 200행, v2는 학습 400행 추가입니다. 같은 검증 데이터로 비교합니다.
Studio의 **Data**, **Components**, **Environments**를 열어 생성된 자산을 확인하세요.

In [ ]:
assets = register_assets(client, settings)
print(assets['environment'])
print(assets['components'])
make_dataset('1').head()

## 2. 기본 모델 학습
제출은 Compute Instance에서 하지만 실제 Prepare/Train/Evaluate는 Compute Cluster에서 실행됩니다.
아래 URL을 열고 그래프와 `evaluate_gate` 자식 Job의 Metrics를 관찰하세요.

In [ ]:
baseline = submit(client, settings, baseline_run, '1', alpha=1.0, max_rmse=3.0)
print(baseline['studio_url'])

In [ ]:
wait_for_run(client, settings, baseline_run)
baseline_report = download_report(client, settings, baseline_run)
assert baseline_report['approved'] is True and baseline_report['rmse'] <= 3.0
baseline_report

## 3. 나쁜 모델은 실제로 실패해야 합니다
**다음 셀의 RuntimeError는 의도된 결과**입니다. Studio에서 `QUALITY_GATE_FAILED`를 확인한 후 다음 셀로 이동합니다.
다른 원인(인증, 네트워크, 환경 오류)의 실패를 품질 게이트 통과 실습으로 간주하지 마세요.

In [ ]:
bad = submit(client, settings, bad_run, '1', alpha=1000000.0, max_rmse=3.0)
print(bad['studio_url'])
wait_for_run(client, settings, bad_run)

In [ ]:
bad_state = snapshot_run(client, settings, bad_run)
bad_report = download_report(client, settings, bad_run)
assert bad_state['status'] == 'Failed'
assert any(c['display_name'] == 'evaluate_gate' and c['status'] == 'Failed' for c in bad_state['children'])
assert bad_report['approved'] is False and bad_report['rmse'] > 3.0
bad_report

다음 등록 셀도 **Registration blocked** 오류가 나야 합니다. 오류를 확인한 뒤 기본 모델 등록으로 진행하세요.

In [ ]:
register_model(client, settings, bad_run, rejected_version)

## 4. 승인 모델 등록 → blue 배포
**여기부터 추론 VM 비용이 발생합니다.** Studio의 Models에서 lineage를 보고 Endpoints에서 배포 상태를 관찰합니다.
`--deployment`에 해당하는 직접 호출과 트래픽 라우팅 호출은 서로 다른 검증입니다.

In [ ]:
registered1 = register_model(client, settings, baseline_run, version1)
print(registered1['id'])
deploy(client, settings, version1, 'blue')
blue_direct = invoke(client, settings, 'blue')
set_traffic(client, settings, 'blue')
blue_routed = invoke(client, settings)
blue_routed

## 5. 데이터 v2로 재학습
같은 파이프라인을 다시 사용합니다. 새 검증 데이터로 바꿔 유리한 결과를 만드는 것이 아니라, 고정 검증 세트로 비교합니다.

In [ ]:
retrain = submit(client, settings, retrain_run, '2', alpha=0.1, max_rmse=3.0)
print(retrain['studio_url'])
wait_for_run(client, settings, retrain_run)
retrain_report = download_report(client, settings, retrain_run)
assert retrain_report['validation_sha256'] == baseline_report['validation_sha256']
assert retrain_report['approved'] is True
print({'baseline_rmse': baseline_report['rmse'], 'retrain_rmse': retrain_report['rmse']})
registered2 = register_model(client, settings, retrain_run, version2)

## 6. green 검증 → 전환 → 롤백
green을 직접 테스트하는 동안 기본 endpoint는 blue를 유지합니다. 사람이 비교하고 트래픽 전환을 승인하는 실습입니다.

In [ ]:
deploy(client, settings, version2, 'green')
green_direct = invoke(client, settings, 'green')
green_direct

In [ ]:
set_traffic(client, settings, 'green')
green_routed = invoke(client, settings)
set_traffic(client, settings, 'blue')
rollback_routed = invoke(client, settings)
print({'green': green_routed['predictions'], 'rollback_blue': rollback_routed['predictions']})

## 7. 비용 정리
Endpoint 삭제는 blue/green 추론 VM도 제거합니다. 개발 Instance도 중지합니다. **이 Notebook의 Instance를 중지하면 연결이 끊길 수 있습니다.**
Studio에서 Instance Stopped, Cluster 실제 노드 0, Endpoint 삭제를 확인하세요. Storage/디스크/Private Endpoint 비용은 별도로 남습니다.
임시 runner VM을 사용한 강사는 `docs/troubleshooting.md`의 별도 정리도 수행합니다.

In [ ]:
cleanup_runtime(client, settings, delete_endpoint=True)